# 04 — Reconcile the manual event catalogues and define reduced-time associations

This notebook reconstructs the Falcon 9 infrasound-event catalogue from the
manual Antelope `dbpick` arrivals preserved by the MATLAB consolidation script:

```text
/Users/thompsong/Developer/KSCRocketSeismology/matlab/fireball/
consolidate_spacex_matlab_legacy_data_epoch_complete.m
```

The principal legacy exports are:

```text
manual_arrival_picks.csv
legacy_infrasound_event_catalogue.csv
legacy_catalog_157_events.csv
```

The two legacy catalogue products contain 153 and 157 event windows,
respectively. They are treated here as provenance products rather than as an
authoritative final count.

The notebook first examines the original, unreduced pick times so that the
legacy discrepancy remains transparent. It then applies predicted
source-to-sensor differential travel-time corrections, associates the reduced
picks, examines association-window sensitivity, and writes the production
candidate and decision tables.

Unix epoch seconds are the authoritative interchange representation. Legacy
channel names `HD1`–`HD3` are mapped to the current StationXML/stream names
`DD1`–`DD3` without modifying the archived pick table.

**Channel convention:** archived manual picks may use HD1/HD2/HD3. Notebook 04 preserves those values in `manual_pick_channel`, maps them once to DD1/DD2/DD3, and uses only the D* codes for geometry, association, outputs, figures, and downstream analysis.


## Objectives

1. Verify that millisecond-level timing is preserved in the manual picks.
2. Document same-channel and global inter-pick spacing in the unreduced data.
3. Compare transparent raw-time clustering rules without selecting a
   threshold merely because it reproduces a legacy event count.
4. Compare the 153- and 157-event legacy catalogue products and identify
   unsupported, split, merged, overlapping, and unmatched windows.
5. Reduce each accepted pick to the DD2 reference time using the surveyed
   geometry and meteorologically estimated effective sound speed.
6. Test the sensitivity of reduced-time associations to the association
   window and adopt the smallest stable, waveform-supported value.
7. Require at least two distinct infrasound channels for a candidate event.
8. Record the targeted waveform decisions that convert 154 reduced-time
   candidates into the final 153-event detection catalogue.
9. Hand the catalogue to later notebooks for baseline correction, waveform
   quality measurement, and refined array analysis.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.dates import DateFormatter

project_root = Path.cwd().resolve()
if project_root.name == "notebooks2":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from modules import project_config as config

PICK_FILE = config.LEGACY_EXPORT_DIR / "manual_arrival_picks.csv"
CATALOG_153_FILE = config.LEGACY_EXPORT_DIR / "legacy_infrasound_event_catalogue.csv"
CATALOG_157_FILE = config.LEGACY_EXPORT_DIR / "legacy_catalog_157_events.csv"
ANALYSIS_CONFIG_FILE = config.OUTPUT_DIR / "02_analysis_configuration.json"
GEOMETRY_FILE = config.OUTPUT_DIR / "02_bchh_geometry.csv"
WEATHER_FILE = config.OUTPUT_DIR / "03_weather_acoustic_summary.csv"

plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 120,
})

## 2. Load picks and both legacy catalogues

Epoch-second columns are preferred because they preserve the original
sub-second timing without depending on CSV datetime formatting.

In [ ]:
for required_file in [
    PICK_FILE,
    CATALOG_153_FILE,
    CATALOG_157_FILE,
]:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Required export not found: {required_file}\n"
            "Rerun consolidate_spacex_matlab_legacy_data_epoch_complete.m"
        )

picks_all = pd.read_csv(PICK_FILE)
catalog_153 = pd.read_csv(CATALOG_153_FILE)
catalog_157 = pd.read_csv(CATALOG_157_FILE)

required_pick_columns = {
    "pickIndex",
    "arrivalTimeEpochS",
    "channel",
    "phase",
}
missing = required_pick_columns.difference(picks_all.columns)
if missing:
    raise KeyError(
        f"Pick table is missing required columns: {sorted(missing)}"
    )

required_153_columns = {
    "eventNumber",
    "firstArrivalEpochS",
    "lastArrivalEpochS",
}
missing = required_153_columns.difference(catalog_153.columns)
if missing:
    raise KeyError(
        "153-event catalogue is missing required columns: "
        f"{sorted(missing)}"
    )

required_157_columns = {
    "catalogEventNumber",
    "onTimeEpochS",
    "offTimeEpochS",
}
missing = required_157_columns.difference(catalog_157.columns)
if missing:
    raise KeyError(
        "157-event catalogue is missing required columns: "
        f"{sorted(missing)}"
    )

picks_all["arrival_time"] = pd.to_datetime(
    picks_all["arrivalTimeEpochS"],
    unit="s",
    utc=True,
)
catalog_153["first_arrival_time"] = pd.to_datetime(
    catalog_153["firstArrivalEpochS"],
    unit="s",
    utc=True,
)
catalog_153["last_arrival_time"] = pd.to_datetime(
    catalog_153["lastArrivalEpochS"],
    unit="s",
    utc=True,
)
catalog_157["on_time"] = pd.to_datetime(
    catalog_157["onTimeEpochS"],
    unit="s",
    utc=True,
)
catalog_157["off_time"] = pd.to_datetime(
    catalog_157["offTimeEpochS"],
    unit="s",
    utc=True,
)

picks_all["channel"] = (
    picks_all["channel"]
    .astype(str)
    .str.upper()
    .str.strip()
)
picks_all["phase_normalized"] = (
    picks_all["phase"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

if "isDeleted" in picks_all.columns:
    picks_all["is_deleted"] = (
        picks_all["isDeleted"]
        .astype(str)
        .str.lower()
        .isin(["true", "1"])
    )
else:
    picks_all["is_deleted"] = (
        picks_all["phase_normalized"] == "del"
    )

print(f"Manual picks: {len(picks_all)}")
print(f"153-event catalogue: {len(catalog_153)}")
print(f"157-event catalogue: {len(catalog_157)}")

## 3. Define the accepted infrasound-pick working set

In [ ]:
# Use the full temporal extent of the 157-event catalogue, with a small
# margin, rather than a hard-coded accident interval.
margin_s = 1.0
analysis_start_epoch_s = catalog_157["onTimeEpochS"].min() - margin_s
analysis_end_epoch_s = catalog_157["offTimeEpochS"].max() + margin_s

# The archived Antelope picks use historical H* pressure-channel codes, whereas
# the calibrated waveform, StationXML, geometry, and all production analysis
# use D* codes. Preserve the archived value only as provenance and map once to
# the canonical D* channel code used everywhere downstream.
PRESSURE_CHANNEL_ALIASES = {
    "HD1": "DD1",
    "HD2": "DD2",
    "HD3": "DD3",
    "DD1": "DD1",
    "DD2": "DD2",
    "DD3": "DD3",
}

picks_all["manual_pick_channel"] = picks_all["channel"]
picks_all["analysis_channel"] = picks_all["manual_pick_channel"].map(
    PRESSURE_CHANNEL_ALIASES
)

accepted_infrasound = picks_all.loc[
    picks_all["arrivalTimeEpochS"].between(
        analysis_start_epoch_s,
        analysis_end_epoch_s,
        inclusive="both",
    )
    & picks_all["analysis_channel"].notna()
    & picks_all["phase_normalized"].eq("n")
    & ~picks_all["is_deleted"]
].copy()

# From this point onward, D* is the operational channel identity. Retain
# manual_pick_channel only for audit/provenance.
accepted_infrasound["channel"] = accepted_infrasound["analysis_channel"]

accepted_infrasound = accepted_infrasound.sort_values(
    ["arrivalTimeEpochS", "analysis_channel", "pickIndex"]
).reset_index(drop=True)

print(
    "Accepted manual infrasound picks:",
    len(accepted_infrasound),
)
print(
    "Canonical analysis channels:",
    sorted(accepted_infrasound["analysis_channel"].unique()),
)


## 4. Verify sub-second precision

This is a direct check that epoch-second values preserve the millisecond
timing seen in the MATLAB catalog display.

In [ ]:
precision_check = accepted_infrasound[
    [
        "pickIndex",
        "arrivalTimeEpochS",
        "arrival_time",
        "channel",
    ]
].head(20).copy()

precision_check["fractional_second"] = (
    precision_check["arrivalTimeEpochS"] % 1.0
)

display(precision_check)
print(
    "Unique fractional seconds among accepted picks:",
    accepted_infrasound["arrivalTimeEpochS"]
    .mod(1.0)
    .round(6)
    .nunique(),
)

## 5. Same-channel inter-pick spacing

In [ ]:
same_channel_parts = []

for channel, group in accepted_infrasound.groupby("channel"):
    group = group.sort_values("arrivalTimeEpochS").copy()
    group["same_channel_gap_s"] = (
        group["arrivalTimeEpochS"].diff()
    )
    same_channel_parts.append(group)

picks_with_same_channel_gaps = pd.concat(
    same_channel_parts,
    ignore_index=True,
).sort_values(["channel", "arrivalTimeEpochS"])

same_channel_gap_summary = (
    picks_with_same_channel_gaps
    .dropna(subset=["same_channel_gap_s"])
    .groupby("channel")["same_channel_gap_s"]
    .agg(
        count="count",
        minimum_s="min",
        p01_s=lambda x: x.quantile(0.01),
        p05_s=lambda x: x.quantile(0.05),
        p10_s=lambda x: x.quantile(0.10),
        median_s="median",
        p90_s=lambda x: x.quantile(0.90),
        p95_s=lambda x: x.quantile(0.95),
        maximum_s="max",
    )
    .reset_index()
)

display(same_channel_gap_summary)

same_channel_gap_summary.to_csv(
    config.OUTPUT_DIR / "04_same_channel_gap_summary.csv",
    index=False,
)
picks_with_same_channel_gaps.to_csv(
    config.OUTPUT_DIR / "04_accepted_picks_with_same_channel_gaps.csv",
    index=False,
)

In [ ]:
gap_values = (
    picks_with_same_channel_gaps["same_channel_gap_s"]
    .dropna()
)
gap_values = gap_values[gap_values > 0]

fig, ax = plt.subplots(figsize=(8.0, 4.8))
ax.hist(
    gap_values,
    bins=np.logspace(
        np.log10(gap_values.min()),
        np.log10(gap_values.max()),
        80,
    ),
)
ax.set_xscale("log")
ax.set_xlabel("Time since previous pick on same channel (s)")
ax.set_ylabel("Number of picks")
ax.set_title("Same-channel inter-pick spacing")
ax.grid(True, which="both", alpha=0.25)

fig.savefig(
    config.FIGURE_DIR / "04_same_channel_gap_histogram.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 6. Global inter-pick spacing

In [ ]:
accepted_sorted = accepted_infrasound.sort_values(
    ["arrivalTimeEpochS", "channel", "pickIndex"]
).reset_index(drop=True)

accepted_sorted["global_gap_s"] = (
    accepted_sorted["arrivalTimeEpochS"].diff()
)

display(
    accepted_sorted["global_gap_s"]
    .dropna()
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

accepted_sorted.to_csv(
    config.OUTPUT_DIR / "04_accepted_infrasound_picks_with_global_gaps.csv",
    index=False,
)

In [ ]:
global_gaps = accepted_sorted["global_gap_s"].dropna()
global_gaps = global_gaps[global_gaps > 0]

fig, ax = plt.subplots(figsize=(8.0, 4.8))
ax.hist(
    global_gaps,
    bins=np.logspace(
        np.log10(global_gaps.min()),
        np.log10(global_gaps.max()),
        100,
    ),
)
ax.set_xscale("log")
ax.axvline(0.1, linestyle="--", linewidth=1.2, label="0.1 s")
ax.set_xlabel("Time since previous accepted pick (s)")
ax.set_ylabel("Number of gaps")
ax.set_title("Global inter-pick spacing")
ax.grid(True, which="both", alpha=0.25)
ax.legend()

fig.savefig(
    config.FIGURE_DIR / "04_global_pick_gap_histogram.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 7. Transparent clustering-rule comparison

In [ ]:
def cluster_by_consecutive_gap(
    picks: pd.DataFrame,
    threshold_s: float,
) -> pd.DataFrame:
    work = picks.sort_values(
        ["arrivalTimeEpochS", "channel", "pickIndex"]
    ).copy()
    gaps = work["arrivalTimeEpochS"].diff()
    work["cluster_id"] = (
        gaps.isna() | gaps.gt(threshold_s)
    ).cumsum()
    return work


def cluster_by_first_pick_window(
    picks: pd.DataFrame,
    threshold_s: float,
) -> pd.DataFrame:
    work = picks.sort_values(
        ["arrivalTimeEpochS", "channel", "pickIndex"]
    ).copy()

    cluster_ids = np.empty(len(work), dtype=int)
    cluster_id = 0
    cluster_start = None

    for i, pick_epoch_s in enumerate(work["arrivalTimeEpochS"]):
        if cluster_start is None:
            cluster_id += 1
            cluster_start = pick_epoch_s
        elif pick_epoch_s - cluster_start > threshold_s:
            cluster_id += 1
            cluster_start = pick_epoch_s

        cluster_ids[i] = cluster_id

    work["cluster_id"] = cluster_ids
    return work


def summarize_clusters(
    clustered_picks: pd.DataFrame,
    minimum_distinct_channels: int = 2,
) -> pd.DataFrame:
    summary = (
        clustered_picks
        .groupby("cluster_id")
        .agg(
            first_pick_epoch_s=("arrivalTimeEpochS", "min"),
            last_pick_epoch_s=("arrivalTimeEpochS", "max"),
            pick_count=("pickIndex", "count"),
            distinct_channel_count=("channel", "nunique"),
            channels=(
                "channel",
                lambda x: ",".join(sorted(set(x))),
            ),
            pick_indices=(
                "pickIndex",
                lambda x: ",".join(map(str, x)),
            ),
        )
        .reset_index()
    )

    summary["duration_s"] = (
        summary["last_pick_epoch_s"]
        - summary["first_pick_epoch_s"]
    )
    summary["first_pick_time"] = pd.to_datetime(
        summary["first_pick_epoch_s"],
        unit="s",
        utc=True,
    )
    summary["last_pick_time"] = pd.to_datetime(
        summary["last_pick_epoch_s"],
        unit="s",
        utc=True,
    )
    summary["qualifies_as_event"] = (
        summary["distinct_channel_count"]
        >= minimum_distinct_channels
    )
    return summary

In [ ]:
thresholds_s = np.round(
    np.arange(0.02, 0.301, 0.002),
    3,
)

sweep_rows = []

algorithms = {
    "consecutive_gap": cluster_by_consecutive_gap,
    "first_pick_window": cluster_by_first_pick_window,
}

for algorithm_name, algorithm in algorithms.items():
    for threshold_s in thresholds_s:
        clustered = algorithm(
            accepted_infrasound,
            threshold_s,
        )
        summary = summarize_clusters(clustered)
        qualifying = summary.loc[
            summary["qualifies_as_event"]
        ]

        sweep_rows.append({
            "algorithm": algorithm_name,
            "threshold_s": threshold_s,
            "total_clusters": len(summary),
            "events_at_least_two_channels": len(qualifying),
            "events_all_three_channels": int(
                (summary["distinct_channel_count"] == 3).sum()
            ),
            "median_event_duration_s": (
                qualifying["duration_s"].median()
            ),
            "maximum_event_duration_s": (
                qualifying["duration_s"].max()
            ),
        })

threshold_sweep = pd.DataFrame(sweep_rows)

closest_to_legacy_counts = (
    threshold_sweep
    .assign(
        difference_from_153=lambda x: (
            x["events_at_least_two_channels"] - 153
        ).abs(),
        difference_from_157=lambda x: (
            x["events_at_least_two_channels"] - 157
        ).abs(),
    )
    .sort_values(
        [
            "difference_from_157",
            "difference_from_153",
            "algorithm",
            "threshold_s",
        ]
    )
    .head(30)
)

display(closest_to_legacy_counts)

threshold_sweep.to_csv(
    config.OUTPUT_DIR / "04_clustering_threshold_sweep.csv",
    index=False,
)

In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 4.8))

for algorithm_name, group in threshold_sweep.groupby("algorithm"):
    ax.plot(
        group["threshold_s"],
        group["events_at_least_two_channels"],
        linewidth=1.2,
        label=algorithm_name.replace("_", " "),
    )

ax.axhline(153, linestyle="--", linewidth=1.0, label="Legacy 153")
ax.axhline(157, linestyle="-.", linewidth=1.0, label="Legacy 157")
ax.axvline(0.1, linestyle=":", linewidth=1.0, label="0.1 s")

ax.set_xlabel("Clustering threshold (s)")
ax.set_ylabel("Clusters with at least two channels")
ax.set_title("Candidate event count versus clustering threshold")
ax.grid(True, alpha=0.25)
ax.legend()

fig.savefig(
    config.FIGURE_DIR / "04_event_count_vs_clustering_threshold.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 8. Production association using reduced pick times

The preceding raw-time sweep is a provenance and sensitivity diagnostic. It
must not define the production association window because the unreduced pick
times include the travel time across the array. In particular, DD1 is expected
to arrive approximately 0.09 s away from DD2 for a source at SLC-40.

Each pick is therefore reduced to the DD2 reference time:

\[
t_{i,\mathrm{red}} = t_i - \frac{r_i-r_{\mathrm{DD2}}}{c_{\mathrm{eff}}},
\]

where \(r_i\) is the surveyed SLC-40–sensor distance and
\(c_{\mathrm{eff}}\) is the effective sound speed derived in Notebook 03.
The correction is used only for association; it is not written into the
continuous waveform data.

In [ ]:
for required_file in [
    ANALYSIS_CONFIG_FILE,
    GEOMETRY_FILE,
    WEATHER_FILE,
]:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Required upstream product not found: {required_file}"
        )

analysis_config = json.loads(ANALYSIS_CONFIG_FILE.read_text())
geometry = pd.read_csv(GEOMETRY_FILE)
weather = pd.read_csv(WEATHER_FILE)

required_geometry_columns = {"channel", "distance_m"}
missing = required_geometry_columns.difference(geometry.columns)
if missing:
    raise KeyError(
        f"Geometry table is missing required columns: {sorted(missing)}"
    )
if "effective_sound_speed_mps" not in weather.columns:
    raise KeyError(
        "Weather summary lacks effective_sound_speed_mps"
    )

REFERENCE_CHANNEL = str(
    analysis_config["infrasound_reference_channel"]
)
EFFECTIVE_SOUND_SPEED_MPS = float(
    weather.iloc[0]["effective_sound_speed_mps"]
)

pressure_geometry = (
    geometry.loc[
        geometry["channel"].isin(
            sorted(accepted_infrasound["analysis_channel"].unique())
        ),
        ["channel", "distance_m"],
    ]
    .drop_duplicates("channel")
    .set_index("channel")
)

missing_channels = set(
    accepted_infrasound["analysis_channel"].unique()
).difference(pressure_geometry.index)
if missing_channels:
    raise KeyError(
        "No surveyed distance for channels: "
        f"{sorted(missing_channels)}"
    )
if REFERENCE_CHANNEL not in pressure_geometry.index:
    raise KeyError(
        f"Reference channel {REFERENCE_CHANNEL} is absent from geometry"
    )

reference_distance_m = float(
    pressure_geometry.at[REFERENCE_CHANNEL, "distance_m"]
)
relative_travel_time_s = (
    (
        pressure_geometry["distance_m"].astype(float)
        - reference_distance_m
    )
    / EFFECTIVE_SOUND_SPEED_MPS
).to_dict()

accepted_reduced = accepted_infrasound.copy()
accepted_reduced["travel_time_correction_s"] = (
    accepted_reduced["analysis_channel"].map(relative_travel_time_s)
)
accepted_reduced["reduced_pick_epoch_s"] = (
    accepted_reduced["arrivalTimeEpochS"]
    - accepted_reduced["travel_time_correction_s"]
)
accepted_reduced["reduced_pick_time"] = pd.to_datetime(
    accepted_reduced["reduced_pick_epoch_s"],
    unit="s",
    utc=True,
)

correction_summary = (
    accepted_reduced[
        ["analysis_channel", "travel_time_correction_s"]
    ]
    .drop_duplicates()
    .sort_values("analysis_channel")
    .reset_index(drop=True)
)

print(f"Reference channel: {REFERENCE_CHANNEL}")
print(
    "Effective sound speed: "
    f"{EFFECTIVE_SOUND_SPEED_MPS:.3f} m/s"
)
display(correction_summary)


In [ ]:
def associate_reduced_time_picks(
    picks: pd.DataFrame,
    *,
    association_window_s: float,
    minimum_channel_count: int = 2,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Associate reduced picks using a forward window from each seed pick.
    work = (
        picks.rename(columns={"analysis_channel": "association_channel"})
        .sort_values(
            ["reduced_pick_epoch_s", "association_channel", "pickIndex"]
        )
        .copy()
    )

    work["reconstructed_event_number"] = 0
    work["association_status"] = "unassigned"
    work["association_residual_s"] = np.nan

    event_rows = []
    next_event_number = 1
    remaining = set(work.index)

    while remaining:
        seed_index = min(
            remaining,
            key=lambda idx: (
                work.at[idx, "reduced_pick_epoch_s"],
                work.at[idx, "pickIndex"],
            ),
        )
        seed_time = float(work.at[seed_index, "reduced_pick_epoch_s"])

        candidate_indices = [
            idx
            for idx in remaining
            if 0.0
            <= work.at[idx, "reduced_pick_epoch_s"] - seed_time
            <= association_window_s
        ]

        candidates = work.loc[candidate_indices].copy()
        candidates["distance_from_seed_s"] = (
            candidates["reduced_pick_epoch_s"] - seed_time
        ).abs()
        selected = (
            candidates.sort_values(
                [
                    "association_channel",
                    "distance_from_seed_s",
                    "pickIndex",
                ]
            )
            .drop_duplicates("association_channel", keep="first")
        )

        if selected["association_channel"].nunique() >= minimum_channel_count:
            center_time = float(selected["reduced_pick_epoch_s"].median())
            selected_indices = list(selected.index)
            work.loc[
                selected_indices, "reconstructed_event_number"
            ] = next_event_number
            work.loc[selected_indices, "association_status"] = "associated"
            work.loc[
                selected_indices, "association_residual_s"
            ] = (
                work.loc[selected_indices, "reduced_pick_epoch_s"]
                - center_time
            )

            event_rows.append({
                "reconstructed_event_number": next_event_number,
                "reduced_event_epoch_s": center_time,
                "first_reduced_pick_epoch_s": float(
                    selected["reduced_pick_epoch_s"].min()
                ),
                "last_reduced_pick_epoch_s": float(
                    selected["reduced_pick_epoch_s"].max()
                ),
                "corrected_pick_span_s": float(
                    selected["reduced_pick_epoch_s"].max()
                    - selected["reduced_pick_epoch_s"].min()
                ),
                "pick_count": len(selected),
                "channel_count": int(
                    selected["association_channel"].nunique()
                ),
                "channels": ",".join(
                    sorted(selected["association_channel"].unique())
                ),
                "pick_indices": ",".join(
                    selected["pickIndex"].astype(int).astype(str)
                ),
                "observed_first_pick_epoch_s": float(
                    selected["arrivalTimeEpochS"].min()
                ),
                "observed_last_pick_epoch_s": float(
                    selected["arrivalTimeEpochS"].max()
                ),
            })

            for idx in selected_indices:
                remaining.remove(idx)
            next_event_number += 1
        else:
            work.at[seed_index, "association_status"] = "unassociated_seed"
            remaining.remove(seed_index)

    events = pd.DataFrame(event_rows)
    if not events.empty:
        events["reduced_event_time"] = pd.to_datetime(
            events["reduced_event_epoch_s"], unit="s", utc=True
        )
        events["observed_first_pick_time"] = pd.to_datetime(
            events["observed_first_pick_epoch_s"], unit="s", utc=True
        )
        events["observed_last_pick_time"] = pd.to_datetime(
            events["observed_last_pick_epoch_s"], unit="s", utc=True
        )

    return work, events

In [ ]:
ASSOCIATION_WINDOWS_S = np.array(
    [0.008, 0.012, 0.016, 0.020, 0.024, 0.030, 0.040, 0.050, 0.060]
)
PRODUCTION_ASSOCIATION_WINDOW_S = 0.040
MINIMUM_CHANNEL_COUNT = 2

window_rows = []
association_results = {}

for window_s in ASSOCIATION_WINDOWS_S:
    assigned, events = associate_reduced_time_picks(
        accepted_reduced,
        association_window_s=float(window_s),
        minimum_channel_count=MINIMUM_CHANNEL_COUNT,
    )
    association_results[float(window_s)] = (assigned, events)
    window_rows.append({
        "association_window_s": float(window_s),
        "reconstructed_event_count": len(events),
        "associated_pick_count": int(
            assigned["association_status"].eq("associated").sum()
        ),
        "unassociated_pick_count": int(
            (~assigned["association_status"].eq("associated")).sum()
        ),
        "three_channel_event_count": int(
            events["channel_count"].eq(3).sum()
        ),
        "two_channel_event_count": int(
            events["channel_count"].eq(2).sum()
        ),
        "median_corrected_pick_span_s": (
            events["corrected_pick_span_s"].median()
        ),
        "p95_corrected_pick_span_s": (
            events["corrected_pick_span_s"].quantile(0.95)
        ),
    })

reduced_time_window_sweep = pd.DataFrame(window_rows)
display(reduced_time_window_sweep)

reduced_time_window_sweep.to_csv(
    config.OUTPUT_DIR / "04_reduced_time_association_window_sweep.csv",
    index=False,
)

fig, ax = plt.subplots(figsize=(8.0, 4.8))
ax.plot(
    reduced_time_window_sweep["association_window_s"],
    reduced_time_window_sweep["reconstructed_event_count"],
    marker="o",
    label="Reduced-time candidates",
)
ax.axhline(153, linestyle="--", linewidth=1.0, label="Legacy 153")
ax.axhline(157, linestyle="-.", linewidth=1.0, label="Legacy 157")
ax.axvline(
    PRODUCTION_ASSOCIATION_WINDOW_S,
    linestyle=":",
    linewidth=1.0,
    label="Adopted 0.040 s",
)
ax.set_xlabel("Reduced-time association window (s)")
ax.set_ylabel("Candidate events with at least two channels")
ax.set_title("Reduced-time association sensitivity")
ax.grid(True, alpha=0.25)
ax.legend()
fig.savefig(
    config.FIGURE_DIR / "04_reduced_time_event_count_vs_window.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

### Choice of the 0.040 s window

The selected value is not chosen to reproduce either legacy count. Candidate
counts stabilize at 154 from 0.040 through 0.060 s. The 0.040 s value is the
smallest value on that plateau.

Only one candidate is added when increasing the window from 0.030 to 0.040 s:
candidate 137 at approximately 13:22:29.928 UTC. Its DD2 and DD3 waveforms are
coherent after the predicted shift (best correlation approximately 0.879),
and its robust stack SNR is approximately 8.0. It is therefore retained.

Increasing the window beyond 0.040 s adds no supported candidates and only
increases the possibility of grouping neighboring physical events.

In [ ]:
production_assignments, production_candidates = association_results[
    PRODUCTION_ASSOCIATION_WINDOW_S
]

production_candidates = production_candidates.copy()
production_candidates["catalogue_decision"] = "retain"
production_candidates["decision_basis"] = (
    "at least two reduced-time-associated infrasound channels"
)
production_candidates["dd2_dd3_best_correlation"] = np.nan
production_candidates["robust_stack_peak_snr"] = np.nan

# Targeted adjudications established during prior manual waveform review.
# They are retained here as provenance for the authoritative catalogue and
# are not outputs of the later S03 validation step. The recorded correlation
# and SNR values are audit metadata, not thresholds applied by this notebook.
targeted_reviews = {
    137: {
        "catalogue_decision": "retain",
        "decision_basis": (
            "threshold-sensitive DD2-DD3 event; coherent waveform"
        ),
        "dd2_dd3_best_correlation": 0.87855,
        "robust_stack_peak_snr": 8.02,
    },
    148: {
        "catalogue_decision": "exclude_ambiguous",
        "decision_basis": (
            "DD1-DD2 only; no DD3 pick or coherent DD3 counterpart"
        ),
        "dd2_dd3_best_correlation": 0.09233,
        "robust_stack_peak_snr": 5.19,
    },
}

for event_number, review in targeted_reviews.items():
    mask = production_candidates["reconstructed_event_number"].eq(
        event_number
    )
    if mask.sum() != 1:
        raise RuntimeError(
            f"Expected exactly one reconstructed event {event_number}; "
            f"found {int(mask.sum())}. Recheck event numbering before "
            "applying the targeted review decision."
        )
    for column, value in review.items():
        production_candidates.loc[mask, column] = value

final_catalogue = production_candidates.loc[
    production_candidates["catalogue_decision"].eq("retain")
].copy()
final_catalogue["final_event_number"] = np.arange(
    1, len(final_catalogue) + 1
)

if len(production_candidates) != 154 or len(final_catalogue) != 153:
    raise RuntimeError(
        "The expected 154-candidate/153-retained result was not reproduced. "
        "Inspect the upstream picks, geometry, and targeted decisions."
    )

production_assignments.to_csv(
    config.OUTPUT_DIR / "04_reduced_time_pick_assignments.csv",
    index=False,
)
production_candidates.to_csv(
    config.OUTPUT_DIR / "04_event_catalogue_decisions.csv",
    index=False,
)
final_catalogue.to_csv(
    config.OUTPUT_DIR / "04_final_153_event_catalogue.csv",
    index=False,
)

print("Reduced-time candidates:", len(production_candidates))
print("Retained detections:", len(final_catalogue))
display(
    production_candidates.loc[
        production_candidates["reconstructed_event_number"].isin(
            sorted(targeted_reviews)
        ),
        [
            "reconstructed_event_number",
            "reduced_event_time",
            "channels",
            "corrected_pick_span_s",
            "dd2_dd3_best_correlation",
            "robust_stack_peak_snr",
            "catalogue_decision",
            "decision_basis",
        ],
    ]
)

## 9. Assign accepted picks directly to the 157 catalogue windows

Picks are matched using epoch seconds and a very small boundary tolerance.
Overlapping windows are not silently resolved; they are flagged.

In [ ]:
CATALOG_BOUNDARY_TOLERANCE_S = 0.001

catalog_157_windows = catalog_157[
    [
        "catalogEventNumber",
        "onTimeEpochS",
        "offTimeEpochS",
        "on_time",
        "off_time",
        "durationS",
    ]
].copy()

pick_assignment_rows = []

for pick in accepted_infrasound.itertuples(index=False):
    matches = catalog_157_windows.loc[
        (
            pick.arrivalTimeEpochS
            >= catalog_157_windows["onTimeEpochS"]
            - CATALOG_BOUNDARY_TOLERANCE_S
        )
        & (
            pick.arrivalTimeEpochS
            <= catalog_157_windows["offTimeEpochS"]
            + CATALOG_BOUNDARY_TOLERANCE_S
        )
    ]

    if len(matches) == 0:
        status = "unmatched"
        matched_event_numbers = ""
    elif len(matches) == 1:
        status = "unique"
        matched_event_numbers = str(
            int(matches.iloc[0]["catalogEventNumber"])
        )
    else:
        status = "ambiguous_overlap"
        matched_event_numbers = ",".join(
            matches["catalogEventNumber"]
            .astype(int)
            .astype(str)
        )

    pick_assignment_rows.append({
        "pickIndex": pick.pickIndex,
        "arrivalTimeEpochS": pick.arrivalTimeEpochS,
        "arrival_time": pick.arrival_time,
        "channel": pick.channel,
        "catalog157_assignment_status": status,
        "catalog157_event_numbers": matched_event_numbers,
        "catalog157_match_count": len(matches),
    })

pick_assignments_157 = pd.DataFrame(pick_assignment_rows)

display(
    pick_assignments_157[
        "catalog157_assignment_status"
    ].value_counts()
)

pick_assignments_157.to_csv(
    config.OUTPUT_DIR / "04_accepted_pick_assignments_to_catalog157.csv",
    index=False,
)

## 10. Summarize pick support for every 157-event window

In [ ]:
event_support_rows = []

for event in catalog_157_windows.itertuples(index=False):
    event_picks = accepted_infrasound.loc[
        (
            accepted_infrasound["arrivalTimeEpochS"]
            >= event.onTimeEpochS - CATALOG_BOUNDARY_TOLERANCE_S
        )
        & (
            accepted_infrasound["arrivalTimeEpochS"]
            <= event.offTimeEpochS + CATALOG_BOUNDARY_TOLERANCE_S
        )
    ].copy()

    event_support_rows.append({
        "catalog157_event_number": int(event.catalogEventNumber),
        "on_time_epoch_s": event.onTimeEpochS,
        "off_time_epoch_s": event.offTimeEpochS,
        "on_time": event.on_time,
        "off_time": event.off_time,
        "catalog_duration_s": event.durationS,
        "pick_count": len(event_picks),
        "distinct_channel_count": event_picks["analysis_channel"].nunique(),
        "channels": ",".join(
            sorted(event_picks["analysis_channel"].unique())
        ),
        "first_pick_epoch_s": (
            event_picks["arrivalTimeEpochS"].min()
            if len(event_picks)
            else np.nan
        ),
        "last_pick_epoch_s": (
            event_picks["arrivalTimeEpochS"].max()
            if len(event_picks)
            else np.nan
        ),
    })

catalog_157_support = pd.DataFrame(event_support_rows)

catalog_157_support["pick_span_s"] = (
    catalog_157_support["last_pick_epoch_s"]
    - catalog_157_support["first_pick_epoch_s"]
)
catalog_157_support["first_pick_minus_on_s"] = (
    catalog_157_support["first_pick_epoch_s"]
    - catalog_157_support["on_time_epoch_s"]
)
catalog_157_support["last_pick_minus_off_s"] = (
    catalog_157_support["last_pick_epoch_s"]
    - catalog_157_support["off_time_epoch_s"]
)
catalog_157_support["has_at_least_two_channels"] = (
    catalog_157_support["distinct_channel_count"] >= 2
)
catalog_157_support["has_all_three_channels"] = (
    catalog_157_support["distinct_channel_count"] == 3
)

display(
    catalog_157_support[
        [
            "pick_count",
            "distinct_channel_count",
            "has_at_least_two_channels",
            "has_all_three_channels",
        ]
    ].describe()
)

display(
    catalog_157_support.loc[
        ~catalog_157_support["has_at_least_two_channels"]
    ]
)

catalog_157_support.to_csv(
    config.OUTPUT_DIR / "04_catalog157_event_pick_support.csv",
    index=False,
)

## 11. Detect overlapping 157-event windows

In [ ]:
catalog_157_sorted = catalog_157_windows.sort_values(
    "onTimeEpochS"
).reset_index(drop=True)

overlap_rows = []

for i in range(len(catalog_157_sorted) - 1):
    current = catalog_157_sorted.iloc[i]
    following = catalog_157_sorted.iloc[i + 1]

    overlap_s = (
        current["offTimeEpochS"]
        - following["onTimeEpochS"]
    )

    if overlap_s >= 0:
        overlap_rows.append({
            "event_a": int(current["catalogEventNumber"]),
            "event_b": int(following["catalogEventNumber"]),
            "event_a_on": current["on_time"],
            "event_a_off": current["off_time"],
            "event_b_on": following["on_time"],
            "event_b_off": following["off_time"],
            "overlap_s": overlap_s,
        })

catalog_157_overlaps = pd.DataFrame(overlap_rows)

print(
    "Number of overlapping/touching adjacent 157-event windows:",
    len(catalog_157_overlaps),
)
display(catalog_157_overlaps)

catalog_157_overlaps.to_csv(
    config.OUTPUT_DIR / "04_catalog157_overlapping_windows.csv",
    index=False,
)

## 12. Compare the 153- and 157-event catalogues

Each 153-event interval is compared with all 157-event intervals. The
overlap duration and midpoint separation are used to identify likely
one-to-one matches, splits, and merges.

In [ ]:
catalog_153_compare = catalog_153[
    [
        "eventNumber",
        "firstArrivalEpochS",
        "lastArrivalEpochS",
        "first_arrival_time",
        "last_arrival_time",
    ]
].copy()

catalog_153_compare["midpoint_epoch_s"] = (
    catalog_153_compare["firstArrivalEpochS"]
    + catalog_153_compare["lastArrivalEpochS"]
) / 2.0

catalog_157_compare = catalog_157_windows.copy()
catalog_157_compare["midpoint_epoch_s"] = (
    catalog_157_compare["onTimeEpochS"]
    + catalog_157_compare["offTimeEpochS"]
) / 2.0

pair_rows = []

for event153 in catalog_153_compare.itertuples(index=False):
    for event157 in catalog_157_compare.itertuples(index=False):
        overlap_start = max(
            event153.firstArrivalEpochS,
            event157.onTimeEpochS,
        )
        overlap_end = min(
            event153.lastArrivalEpochS,
            event157.offTimeEpochS,
        )
        overlap_s = max(0.0, overlap_end - overlap_start)

        midpoint_difference_s = (
            event157.midpoint_epoch_s
            - event153.midpoint_epoch_s
        )

        if overlap_s > 0 or abs(midpoint_difference_s) <= 0.25:
            pair_rows.append({
                "event153": int(event153.eventNumber),
                "event157": int(event157.catalogEventNumber),
                "overlap_s": overlap_s,
                "midpoint_difference_s": midpoint_difference_s,
                "start_difference_s": (
                    event157.onTimeEpochS
                    - event153.firstArrivalEpochS
                ),
                "end_difference_s": (
                    event157.offTimeEpochS
                    - event153.lastArrivalEpochS
                ),
            })

catalogue_pairs = pd.DataFrame(pair_rows)

# Best 157 match for each 153 event:
best_157_for_153 = (
    catalogue_pairs
    .sort_values(
        [
            "event153",
            "overlap_s",
            "midpoint_difference_s",
        ],
        ascending=[True, False, True],
    )
    .groupby("event153", as_index=False)
    .first()
)

# Best 153 match for each 157 event:
best_153_for_157 = (
    catalogue_pairs
    .assign(
        absolute_midpoint_difference_s=lambda x: (
            x["midpoint_difference_s"].abs()
        )
    )
    .sort_values(
        [
            "event157",
            "overlap_s",
            "absolute_midpoint_difference_s",
        ],
        ascending=[True, False, True],
    )
    .groupby("event157", as_index=False)
    .first()
)

display(best_157_for_153.head(20))
display(best_153_for_157.head(20))

catalogue_pairs.to_csv(
    config.OUTPUT_DIR / "04_catalog153_catalog157_candidate_pairs.csv",
    index=False,
)
best_157_for_153.to_csv(
    config.OUTPUT_DIR / "04_catalog153_best_catalog157_match.csv",
    index=False,
)
best_153_for_157.to_csv(
    config.OUTPUT_DIR / "04_catalog157_best_catalog153_match.csv",
    index=False,
)

## 13. Identify likely splits and merges

In [ ]:
meaningful_pairs = catalogue_pairs.loc[
    (catalogue_pairs["overlap_s"] > 0)
    | (catalogue_pairs["midpoint_difference_s"].abs() <= 0.05)
].copy()

split_counts = (
    meaningful_pairs
    .groupby("event153")["event157"]
    .nunique()
    .rename("n_catalog157_matches")
    .reset_index()
)
likely_splits = split_counts.loc[
    split_counts["n_catalog157_matches"] > 1
]

merge_counts = (
    meaningful_pairs
    .groupby("event157")["event153"]
    .nunique()
    .rename("n_catalog153_matches")
    .reset_index()
)
likely_merges = merge_counts.loc[
    merge_counts["n_catalog153_matches"] > 1
]

print("Possible 153 → multiple 157 splits:")
display(likely_splits)

print("Possible multiple 153 → one 157 merges:")
display(likely_merges)

likely_splits.to_csv(
    config.OUTPUT_DIR / "04_possible_catalog153_splits.csv",
    index=False,
)
likely_merges.to_csv(
    config.OUTPUT_DIR / "04_possible_catalog157_merges.csv",
    index=False,
)

## 14. Build the focused waveform-review list

An event is flagged for visual review when any of the following is true:

- fewer than two infrasound channels support a 157-event window;
- the 157 window overlaps or touches another window;
- the event participates in a possible split or merge;
- no close counterpart exists in the 153 catalogue;
- the 153 and 157 window boundaries differ substantially.

In [ ]:
review_reasons = {}

def add_reason(event_number: int, reason: str) -> None:
    review_reasons.setdefault(event_number, set()).add(reason)

for row in catalog_157_support.itertuples(index=False):
    event_number = int(row.catalog157_event_number)

    if row.distinct_channel_count < 2:
        add_reason(event_number, "fewer than two infrasound channels")

    if row.pick_count == 0:
        add_reason(event_number, "no accepted infrasound picks")

    if np.isfinite(row.first_pick_minus_on_s):
        if abs(row.first_pick_minus_on_s) > 0.005:
            add_reason(event_number, "first pick differs from ontime")

    if np.isfinite(row.last_pick_minus_off_s):
        if abs(row.last_pick_minus_off_s) > 0.005:
            add_reason(event_number, "last pick differs from offtime")

if not catalog_157_overlaps.empty:
    for row in catalog_157_overlaps.itertuples(index=False):
        add_reason(int(row.event_a), "overlapping/touching window")
        add_reason(int(row.event_b), "overlapping/touching window")

split_event_157s = set(
    meaningful_pairs.loc[
        meaningful_pairs["event153"].isin(
            likely_splits["event153"]
        ),
        "event157",
    ].astype(int)
)
for event_number in split_event_157s:
    add_reason(event_number, "possible split from 153 catalogue")

for event_number in likely_merges["event157"].astype(int):
    add_reason(event_number, "possible merge relative to 153 catalogue")

matched_157 = set(best_153_for_157["event157"].astype(int))
for event_number in catalog_157["catalogEventNumber"].astype(int):
    if event_number not in matched_157:
        add_reason(event_number, "no close 153-event counterpart")

for row in best_153_for_157.itertuples(index=False):
    if abs(row.start_difference_s) > 0.05:
        add_reason(int(row.event157), "start differs from 153 by >0.05 s")
    if abs(row.end_difference_s) > 0.05:
        add_reason(int(row.event157), "end differs from 153 by >0.05 s")

review_rows = [
    {
        "catalog157_event_number": event_number,
        "reasons": "; ".join(sorted(reasons)),
        "reason_count": len(reasons),
    }
    for event_number, reasons in sorted(review_reasons.items())
]

waveform_review_list = pd.DataFrame(review_rows)

waveform_review_list = waveform_review_list.merge(
    catalog_157_support,
    on="catalog157_event_number",
    how="left",
)

waveform_review_list = waveform_review_list.sort_values(
    [
        "reason_count",
        "catalog157_event_number",
    ],
    ascending=[False, True],
)

print(
    "157-event windows recommended for visual review:",
    len(waveform_review_list),
)
display(waveform_review_list)

waveform_review_list.to_csv(
    config.OUTPUT_DIR / "04_waveform_review_list.csv",
    index=False,
)

## 15. Pick raster with 157-event windows

In [ ]:
channel_y = {"DD1": 3, "DD2": 2, "DD3": 1}

raster = accepted_infrasound.copy()
raster["channel_y"] = raster["analysis_channel"].map(channel_y)

fig, ax = plt.subplots(figsize=(12.0, 4.5))

ax.scatter(
    raster["arrival_time"],
    raster["channel_y"],
    s=14,
    label="Accepted picks",
)

for event in catalog_157_windows.itertuples(index=False):
    ax.axvspan(
        event.on_time,
        event.off_time,
        alpha=0.38,
        color="red",
    )

ax.set_yticks([1, 2, 3])
ax.set_yticklabels(["DD3", "DD2", "DD1"])
ax.set_ylim(0.5, 3.5)
ax.set_xlabel("Time on 1 September 2016 (UTC)")
ax.set_ylabel("Canonical infrasound channel")
ax.set_title("Accepted manual picks and 157-event catalogue windows")
ax.xaxis.set_major_formatter(DateFormatter("%H:%M"))
ax.grid(True, axis="x", alpha=0.25)

fig.savefig(
    config.FIGURE_DIR / "04_accepted_pick_raster_with_catalog157.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## 16. Catalogue decision and quality-analysis handoff

The production sequence established here is:

1. retain accepted manual Antelope `N` picks on the infrasound sensors;
2. map archived HD1–HD3 names to current DD1–DD3 names;
3. reduce each pick to DD2 using surveyed distances and the effective sound
   speed from Notebook 03;
4. associate reduced picks within a forward 0.040 s window;
5. require at least two distinct infrasound channels;
6. obtain 154 candidates;
7. retain threshold-sensitive candidate 137 because DD2 and DD3 are coherent;
8. exclude candidate 148 because it is supported only by DD1 and DD2 and has
   no coherent DD3 counterpart;
9. retain a final detection catalogue of 153 events.

DD1 is retained as supporting information, but it must not veto an event
supported by DD2 and DD3 because it contains fewer accepted picks and exhibits
substantially poorer waveform consistency.

Event detection and waveform quality are deliberately separated. Subsequent
notebooks should measure, for every retained event:

- robust peak signal-to-noise ratio;
- signal-window to noise-window standard-deviation or RMS ratio;
- shifted DD2–DD3 correlation and residual lag;
- pressure amplitude and pulse morphology;
- three-channel array coherence where DD1 is usable.

The current downstream waveform audit gives the following operational quality
subsets of the 153 detections:

| Subset | Criteria | Events |
|---|---|---:|
| Detection catalogue | DD2 and DD3 reduced-time pick support | 153 |
| Usable waveform subset | robust peak SNR >= 3 and DD2–DD3 correlation >= 0.5 | 138 |
| Core waveform subset | robust peak SNR >= 5 and DD2–DD3 correlation >= 0.7 | 107 |
| Marginal detections | retained detections below the usable thresholds | 15 |

The RMS or standard-deviation ratio should remain a continuous diagnostic, not
an additional catalogue-defining cutoff. Short impulses can have high peak SNR
but only moderate windowed RMS, and RMS is strongly dependent on the selected
window duration. Nearby thresholds should be retained in sensitivity tables;
the stated values are operational quality divisions rather than natural event
classes or mathematically unique optima.

Primary outputs from this notebook are:

```text
04_reduced_time_association_window_sweep.csv
04_reduced_time_pick_assignments.csv
04_event_catalogue_decisions.csv
04_final_153_event_catalogue.csv
04_waveform_review_list.csv
```